In [1]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

[nltk_data] Error loading stopwords: <urlopen error [WinError 10060]
[nltk_data]     Ein Verbindungsversuch ist fehlgeschlagen, da die
[nltk_data]     Gegenstelle nach einer bestimmten Zeitspanne nicht
[nltk_data]     richtig reagiert hat, oder die hergestellte Verbindung
[nltk_data]     war fehlerhaft, da der verbundene Host nicht reagiert
[nltk_data]     hat>


In [2]:
df = pd.read_csv("corpus_df.csv", encoding="utf-8")  # replace with your actual file path

NameError: name 'pd' is not defined

In [100]:
custom_sw = german_sw + [  # historical forms
]

In [125]:
docs = df["text"].astype(str).tolist()

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

hdbscan_model = HDBSCAN(
    min_cluster_size=7, ## experiment with this to increase/decrease number of topics
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=custom_sw,
    ngram_range=(1, 2))

umap_model = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

topic_model = BERTopic(
    language="multilingual",
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    verbose=True
)


topics, probs = topic_model.fit_transform(docs)



2026-04-02 17:53:42,779 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

2026-04-02 17:53:59,850 - BERTopic - Embedding - Completed ✓
2026-04-02 17:53:59,851 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-02 17:54:00,389 - BERTopic - Dimensionality - Completed ✓
2026-04-02 17:54:00,391 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-02 17:54:00,406 - BERTopic - Cluster - Completed ✓
2026-04-02 17:54:00,411 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-02 17:54:00,828 - BERTopic - Representation - Completed ✓


In [126]:
topic_info = topic_model.get_topic_info()
topic_model.get_topic_info()[["Topic", "Count", "Name"]]

,Topic,Count,Name
0,-1,70,-1_sr_fr_nov_worden
1,0,30,0_theater_rolle_bühne_mslle
2,1,26,1_kunst_ganz_ausstellung_bild
3,2,25,2_worden_lissabon_nov_paris
4,3,20,3_abendblätter_blätter_berlin_berliner
5,4,19,4_könig_ballon_schweden_claudius
6,5,18,5_kümmel_bürger_kümmel kümmel_verbrechen
7,6,17,6_gr_thl_klaviatur_handbuch
8,7,16,7_mittheilungen_tages mittheilungen_polizeilic...
9,8,15,8_feuer_polizeiliche tages_tages mittheilungen...


In [127]:
topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=False).head(10) ## check the largest topics for interpretability
#topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=True).head(10) ## check the smallest topics for potential noise

,Topic,Count,Name,Representation,Representative_Docs
1,0,30,0_theater_rolle_bühne_mslle,"[theater, rolle, bühne, mslle, wäre, publikum,...",[Theater.\n\nUeber Darstellbarkeit auf der Büh...
2,1,26,1_kunst_ganz_ausstellung_bild,"[kunst, ganz, ausstellung, bild, künstler, bil...",[Kunst-Ausstellung.\n\n(Fortsetzung des im 9te...
3,2,25,2_worden_lissabon_nov_paris,"[worden, lissabon, nov, paris, armee, öffentli...",[Geographische Nachricht von der Insel Helgola...
4,3,20,3_abendblätter_blätter_berlin_berliner,"[abendblätter, blätter, berlin, berliner, stüc...",[Schreiben eines Berliner Einwohners an den He...
5,4,19,4_könig_ballon_schweden_claudius,"[könig, ballon, schweden, claudius, scheint, h...",[Bülletin der öffentlichen Blätter.\n\nAus Sch...
6,5,18,5_kümmel_bürger_kümmel kümmel_verbrechen,"[kümmel, bürger, kümmel kümmel, verbrechen, wo...",[Polizei-Rapport.\n\nVom 4ten October.\n\nDas ...
7,6,17,6_gr_thl_klaviatur_handbuch,"[gr, thl, klaviatur, handbuch, schriften, hitz...","[Interessante Schriften, welche in der Buchhan..."
8,7,16,7_mittheilungen_tages mittheilungen_polizeilic...,"[mittheilungen, tages mittheilungen, polizeili...",[Polizeiliche Tages-Mittheilungen.\n\nBeim Nac...
9,8,15,8_feuer_polizeiliche tages_tages mittheilungen...,"[feuer, polizeiliche tages, tages mittheilunge...",[Polizeiliche Tages-Mittheilungen.\n\nIn einem...
10,9,14,9_herr_jonas_thüringer_spricht,"[herr, jonas, thüringer, spricht, petre, spric...","[Anekdote.\n\nEin mecklenburgischer Landmann, ..."


In [128]:
fig = topic_model.visualize_topics()
fig.show()

In [129]:
fig = topic_model.visualize_barchart()
fig.show()

fig = topic_model.visualize_heatmap()
fig.show()

fig = topic_model.visualize_hierarchy()
fig.show()

fig = topic_model.visualize_documents(docs)
fig.show()

In [130]:
timestamps = df["date"]
timestamps = pd.to_datetime(df["date"])
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    nr_bins=20  # adjust depending on corpus size
)


20it [00:02,  7.71it/s]


In [131]:
fig = topic_model.visualize_topics_over_time(topics_over_time)
fig.show()

In [113]:
classes = df["author"]

In [114]:
topics_per_class = topic_model.topics_per_class(
    docs,
    classes
)

62it [00:04, 13.30it/s]


In [115]:
fig = topic_model.visualize_topics_per_class(topics_per_class)
fig.show()

In [ ]:
ToDO:

- check the largest topics for interpretability
- check the time evolution of topics
- have a way to give the top texts per topic for interpretation
